# R9 — Insight memo (HOS-017 vs HOS-022)

Two hospitals' rejection rates compared — overall, then split by claim type — because a single aggregate number told a misleading story once someone looked underneath it.

* **Decided** = approved + rejected + settled. A claim still at `submitted` has not been decided and belongs on **neither** side of the fraction.
* **Rejection rate** = rejected ÷ decided.
* The `r9_paradox` table carries one row per hospital × claim type (`ALL` = the overall comparison), `HOS-017` and `HOS-022` only.
* The memo cell at the end names the statistical effect behind any reversal by its standard textbook name.

In [0]:
%sql
-- Which claim types exist in the data (for the r9_paradox claim_type split)
SELECT DISTINCT claim_type FROM claims.analytics.fact_claims ORDER BY claim_type;

In [0]:
%sql
-- r9_paradox: overall (claim_type = 'ALL') + per claim type, HOS-017 and HOS-022 only
CREATE OR REPLACE TABLE claims.analytics.r9_paradox AS
WITH decided AS (
  SELECT
    h.hospital_id,
    f.claim_type,
    COUNT(*)                                               AS decided_count,
    SUM(CASE WHEN f.status = 'rejected' THEN 1 ELSE 0 END) AS rejected_count
  FROM claims.analytics.fact_claims f
  JOIN claims.analytics.dim_hospital h ON f.hospital_sk = h.hospital_sk
  WHERE h.hospital_id IN ('HOS-017', 'HOS-022')
    AND f.status IN ('approved', 'rejected', 'settled')
  GROUP BY h.hospital_id, f.claim_type
)
SELECT
  hospital_id,
  claim_type,
  CAST(decided_count   AS INT) AS decided_count,
  CAST(rejected_count  AS INT) AS rejected_count,
  ROUND(rejected_count / decided_count, 4) AS rejection_rate
FROM (
  -- per claim type
  SELECT hospital_id, claim_type, decided_count, rejected_count FROM decided
  UNION ALL
  -- overall comparison (ALL claim types)
  SELECT hospital_id, 'ALL' AS claim_type, SUM(decided_count), SUM(rejected_count)
  FROM decided
  GROUP BY hospital_id
)
ORDER BY hospital_id, claim_type;

In [0]:
%sql
-- The comparison, ready to read: headline number first, then the split underneath it
SELECT * FROM claims.analytics.r9_paradox ORDER BY claim_type, hospital_id;

## Insight memo — HOS-017 vs HOS-022 (rejection rates)

### Question

Two hospitals — **HOS-017** (Coimbatore Care Hospital 17) and **HOS-022** (Pune Care Hospital 22) — need their rejection rates compared. Rejection rate is **rejected ÷ decided** claims (decided = approved + rejected + settled; a claim still at `submitted` is on neither side of the fraction).

### Overall

| Hospital | Decided | Rejected | Rejection rate |
| --- | --- | --- | --- |
| HOS-017 | 100 | 21 | **0.21** |
| HOS-022 | 100 | 11 | **0.11** |

On the headline number, **HOS-017 looks almost twice as bad as HOS-022** (21% vs 11%).

### Split by claim type

| Claim type | HOS-017 decided → rejected (rate) | HOS-022 decided → rejected (rate) | Who is worse |
| --- | --- | --- | --- |
| cashless | 20 → 1 (**0.05**) | 80 → 5 (**0.0625**) | HOS-022 |
| reimbursement | 80 → 20 (**0.25**) | 20 → 6 (**0.30**) | HOS-022 |

Within **both** claim types, **HOS-022 is actually the worse hospital** — the headline number had it backwards.

### Why the overall number misleads

The reversal is driven by the hospitals' **claim-type mix**, not by their rejection behaviour: HOS-017's decided claims are 80% reimbursement (the high-rejection type, ~26% average rejection) while HOS-022's are 80% cashless (the low-rejection type, ~6% average rejection). Aggregating over an uneven mix makes the hospital with the *safer* mix look safer regardless of how it handles each claim type. The statistical effect behind this reversal — an association that reverses direction when data is aggregated over a confounding subgroup variable — is its standard textbook name: **Simpson's paradox** (the confounder here is claim type).

### Recommendation

Never compare the two hospitals on the aggregate number alone. Compare **like with like — within claim type** (where HOS-022 is worse on both), or use a mix-adjusted (standardised) rate if a single number is required. The compliance review should treat `claims.analytics.r9_paradox` (overall row `ALL` + per-claim-type rows) as the reporting basis.